# Lab Assignment 4

MST Dependency Parsing and Training

**Name:** Akshat  
**Roll number:** 12340160

**Total: 20 marks**

## Objective
Implement core components of an MST dependency parser, analyse a Chu-Liu/Edmonds cycle, and perform one structured-perceptron update using gold and predicted dependency trees.

## Submission
- Complete all `TODO` sections and all markdown answers.
- Do not modify the supplied data or expected-test cells.
- Run **Kernel > Restart & Run All** before submitting the completed `.ipynb` file.

## Valid dependency tree conditions
Each non-root word has exactly one incoming head; `ROOT` has none; all words are reachable from `ROOT`; and no directed cycle occurs.

In [28]:
from itertools import product
from collections import defaultdict, Counter
import pandas as pd
import matplotlib.pyplot as plt

## Task 1: Validate Dependency Trees (2 marks)

Implement `is_valid_tree()` to verify whether a set of dependency arcs forms a valid rooted dependency tree.

A valid dependency tree must satisfy all of the following conditions:

1. `ROOT` has no incoming arc.
2. Every non-root word has exactly one incoming head.
3. No word can be its own head (no self-loop).
4. The graph must not contain a directed cycle.
5. Every word must be reachable from `ROOT`.
6. The number of arcs must be exactly `number_of_tokens - 1`.

### Your task

Complete the `is_valid_tree()` function below.

Do not change the test cases.

Your function should return:

- `True` for a valid dependency tree.
- `False` for an invalid dependency structure.

You should ensure that the function correctly detects duplicate heads, missing heads, cycles, self-loops, disconnected components, and arcs entering `ROOT`.

In [29]:
def is_valid_tree(tokens, arcs, root="ROOT"):
    non_root = [w for w in tokens if w != root]

    # condition 6: exactly one arc per non-root word
    if len(arcs) != len(non_root):
        return False

    head_of = {}
    for head, dep in arcs:
        if dep == root:            # condition 1, nothing may point at ROOT
            return False
        if head == dep:            # condition 3, no self loop
            return False
        if dep in head_of:         # condition 2, a word cannot take two heads
            return False
        head_of[dep] = head

    # every non-root word must have received a head
    if set(head_of) != set(non_root):
        return False

    # conditions 4 and 5: walking up the heads from any word must land on ROOT
    for word in non_root:
        seen = set()
        current = word
        while current != root:
            if current in seen or current not in head_of:
                return False
            seen.add(current)
            current = head_of[current]

    return True

test_tokens = ["ROOT", "A", "B", "C"]
valid = [("ROOT","A"), ("A","B"), ("A","C")]
cycle = [("ROOT","A"), ("B","C"), ("C","B")]
duplicate = [("ROOT","A"), ("A","B"), ("C","B")]
print(is_valid_tree(test_tokens, valid))      # Expected: True
print(is_valid_tree(test_tokens, cycle))      # Expected: False
print(is_valid_tree(test_tokens, duplicate))  # Expected: False

True
False
False


## Task 2: Exhaustive MST Reference Parser (3 marks)

Implement an exhaustive reference parser for a small dependency graph.

For every non-root word:

1. Enumerate all possible incoming heads.
2. Construct candidate dependency trees.
3. Use `is_valid_tree()` from Task 1 to reject invalid structures.
4. Compute the total score of every valid tree.
5. Return the highest-scoring tree and its score.

The score of a tree is:

$$
Score(G)=\sum_{(h,d)\in G} Score(h\rightarrow d)
$$

where:

- $h$ = head
- $d$ = dependent
- $G$ = dependency tree

### Your task

Complete:

```python
exhaustive_mst(tokens, score)

In [30]:

tokens = ["ROOT", "Researchers", "analyze", "data", "carefully"]

score = {
    ("ROOT","Researchers"):1,
    ("ROOT","analyze"):7,
    ("ROOT","data"):1,
    ("ROOT","carefully"):1,

    ("Researchers","analyze"):2,
    ("Researchers","data"):2,
    ("Researchers","carefully"):2,

    ("analyze","Researchers"):6,
    ("analyze","data"):7,
    ("analyze","carefully"):6,

    ("data","Researchers"):2,
    ("data","analyze"):3,
    ("data","carefully"):3,

    ("carefully","Researchers"):2,
    ("carefully","analyze"):3,
    ("carefully","data"):4,
}

def exhaustive_mst(tokens, score, root="ROOT"):
    non_root = [w for w in tokens if w != root]

    # 1. every non-root word may take any other word as its head
    choices = [[h for h in tokens if h != dep] for dep in non_root]

    best_score = float("-inf")
    best_arcs = None

    # 2. one candidate tree per combination of heads
    for heads in product(*choices):
        candidate = list(zip(heads, non_root))

        # 3. throw away anything that is not a rooted tree
        if not is_valid_tree(tokens, candidate, root):
            continue

        # 4. score the surviving tree
        total = sum(score[arc] for arc in candidate)

        # 5. remember the best one
        if total > best_score:
            best_score = total
            best_arcs = candidate

    return best_score, best_arcs

best_score, best_arcs = exhaustive_mst(tokens, score)

print("Best score:", best_score)
print("Best arcs:", best_arcs)

Best score: 26
Best arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]


## Task 3: Chu-Liu/Edmonds — Best Incoming Edges and Cycle Detection (3 marks)

The Chu-Liu/Edmonds algorithm starts by selecting the highest-scoring incoming edge for every non-root node.

For each node $v$:

$$
parent(v)
=
\arg\max_{u\neq v} Score(u\rightarrow v)
$$

If the selected edges form a valid tree, the algorithm is finished.

However, if the selected edges contain a directed cycle, the cycle must be detected and contracted before continuing.

### Your task

Complete the following two functions:

1. `best_incoming()`
2. `find_cycle()`

`best_incoming()` should select the highest-scoring incoming edge for every non-root node.

`find_cycle()` should:

- return one directed cycle as a list of nodes if a cycle exists;
- return `None` if no cycle exists.

### Questions

After running the code:

1. What edges were selected by the greedy incoming-edge step?
2. Which nodes form the cycle?
3. Why is the greedy result not a valid dependency tree?
4. Why can we not simply accept the greedy result as the MST?

In [31]:
cycle_tokens = ["ROOT", "A", "B", "C"]

cycle_score = {
    ("ROOT","A"):5,
    ("ROOT","B"):4,
    ("ROOT","C"):3,

    ("A","B"):10,
    ("B","A"):10,

    ("A","C"):7,
    ("B","C"):8,

    ("C","A"):2,
    ("C","B"):2,
}


def best_incoming(tokens, score, root="ROOT"):
    arcs = []

    for dep in tokens:
        if dep == root:                 # ROOT never takes an incoming edge
            continue

        candidates = [
            (head, dep)
            for head in tokens
            if head != dep and (head, dep) in score
        ]
        if not candidates:
            continue

        arcs.append(max(candidates, key=lambda arc: score[arc]))

    return arcs


def find_cycle(arcs):
    head_of = {dep: head for head, dep in arcs}

    # start at each node and keep following the head pointers
    for start in head_of:
        path = []
        current = start
        while current in head_of:
            if current in path:
                return path[path.index(current):]
            path.append(current)
            current = head_of[current]

    return None


greedy_arcs = best_incoming(cycle_tokens, cycle_score)

print("Greedy arcs:", greedy_arcs)
print("Cycle:", find_cycle(greedy_arcs))

Greedy arcs: [('B', 'A'), ('A', 'B'), ('B', 'C')]
Cycle: ['A', 'B']


**Task 3 explanation:**

**1. Edges selected by the greedy step**

```text
B -> A   score 10
A -> B   score 10
B -> C   score 8
```

Each dependent simply took its single best incoming edge. `A` prefers `B` over `ROOT` because 10 beats 5, `B` prefers `A` over `ROOT` because 10 beats 4, and `C` prefers `B` over `A` and `ROOT` because 8 is the largest of 8, 7 and 3.

**2. Nodes forming the cycle**

`A` and `B`. The head of `A` is `B` and the head of `B` is `A`, so `A -> B -> A` closes on itself.

**3. Why the greedy result is not a valid tree**

Two of the six tree conditions fail. There is a directed cycle between `A` and `B`, and because of that cycle neither `A` nor `B` nor `C` is reachable from `ROOT`. Following the head pointers from `C` gives `C -> B -> A -> B -> ...` and never arrives at `ROOT`. `ROOT` has no dependent at all, so the structure is not connected to the root.

**4. Why we cannot accept the greedy result as the MST**

The greedy step maximises each incoming edge on its own and ignores the constraint that the arcs together have to form a tree. Once a cycle appears, one of its edges has to be given up, and the edge that is cheapest to give up is not decided locally. Dropping `B -> A` costs 10 but lets `ROOT -> A` in for 5, a net loss of 5, while dropping `A -> B` costs 10 and lets `ROOT -> B` in for 4, a net loss of 6. That comparison is exactly what the contraction step of Chu-Liu/Edmonds performs, so the cycle has to be contracted and solved rather than accepted.

## Task 4: Chu-Liu/Edmonds Cycle Contraction (3 marks)

Suppose the greedy step produces the cycle:

$$
A\rightarrow B
$$

and

$$
B\rightarrow A
$$

with:

$$
Score(A\rightarrow B)=10
$$

$$
Score(B\rightarrow A)=10
$$

The total score of the selected cycle is:

$$
CycleScore=10+10=20
$$

When an external edge enters a node inside the cycle, the cycle is contracted into a single **supernode**.

For an external edge $h\rightarrow d$ entering the cycle, calculate the adjusted score as:

$$
\boxed{
AdjustedScore(h\rightarrow d)
=
Score(h\rightarrow d)
+
CycleScore
-
Score(old\_incoming\_edge(d))
}
$$

### Example

For:

$$
ROOT\rightarrow A
$$

we have:

$$
Score(ROOT\rightarrow A)=5
$$

The selected edge entering `A` inside the cycle is:

$$
B\rightarrow A
$$

with score:

$$
Score(B\rightarrow A)=10
$$

Therefore:

$$
AdjustedScore(ROOT\rightarrow A)
=
5+20-10
=
15
$$

### Your task

Complete `entering_cycle_table()`.

Create a table for:

- `ROOT -> A`
- `ROOT -> B`

The table must contain:

- candidate edge
- original score
- selected cycle edge being removed
- removed edge score
- total cycle score
- adjusted score

### Interpretation question

Which candidate edge should be selected to enter the contracted cycle?

Explain which original cycle edge will be removed when the contracted tree is expanded back to the original graph.

In [32]:
def entering_cycle_table(score, cycle, selected_cycle_arcs):
    cycle_set = set(cycle)

    # the selected edge that currently enters each node of the cycle
    removed_for = {
        dep: (head, dep)
        for head, dep in selected_cycle_arcs
        if dep in cycle_set
    }

    cycle_total = sum(score[arc] for arc in removed_for.values())

    rows = []
    for (head, dep), value in score.items():
        # keep only edges coming from outside into the cycle
        if dep not in cycle_set or head in cycle_set:
            continue

        removed = removed_for[dep]

        rows.append({
            "outside_head": head,
            "cycle_dependent": dep,
            "original_score": value,
            "removed_cycle_arc": f"{removed[0]} -> {removed[1]}",
            "removed_score": score[removed],
            "cycle_total": cycle_total,
            "adjusted_score": value + cycle_total - score[removed],
        })

    return pd.DataFrame(rows)


selected_cycle_arcs = [
    ("B","A"),
    ("A","B")
]

entering_cycle_table(
    cycle_score,
    ["A", "B"],
    selected_cycle_arcs
)

,outside_head,cycle_dependent,original_score,removed_cycle_arc,removed_score,cycle_total,adjusted_score
0,ROOT,A,5,B -> A,10,20,15
1,ROOT,B,4,A -> B,10,20,14
2,C,A,2,B -> A,10,20,12
3,C,B,2,A -> B,10,20,12


**Task 4 interpretation:**

The table gives these adjusted scores for the edges entering the contracted cycle:

| Candidate edge | Original | Cycle total | Removed cycle edge | Adjusted |
|---|---:|---:|---|---:|
| ROOT -> A | 5 | 20 | B -> A (10) | 15 |
| ROOT -> B | 4 | 20 | A -> B (10) | 14 |

`ROOT -> A` should be selected, because its adjusted score of 15 is the highest.

The adjusted score already contains the price of entering the cycle. An external edge into `A` can only be used if `A` gives up the head it had inside the cycle, so `Score(B -> A) = 10` is subtracted while the rest of the cycle, `Score(A -> B) = 10`, is kept. Entering through `B` would mean subtracting `Score(A -> B) = 10` instead, and since `ROOT -> B` is worth one point less than `ROOT -> A`, that route ends up one point worse.

So when the contracted tree is expanded back, the cycle edge `B -> A` is removed and `A -> B` stays. The final tree is `ROOT -> A`, `A -> B`, `B -> C`, with total score 5 + 10 + 8 = 23.

## Task 5: Implement the Chu-Liu/Edmonds MST Parser (4 marks)

In this task, you will implement the **Chu-Liu/Edmonds algorithm from scratch** for finding the maximum-scoring rooted directed spanning tree.

Do **not** use:

- `networkx.maximum_spanning_arborescence`
- any other library implementation of Chu-Liu/Edmonds

You may use Python data structures such as dictionaries, lists, and sets.

### Algorithm

Your implementation should follow these steps:

#### Step 1: Select the best incoming edge

For every non-root node, select the incoming edge with the highest score.

#### Step 2: Check for cycles

If the selected edges form a valid rooted tree, return them.

If a directed cycle exists, continue.

#### Step 3: Identify the cycle

Find one directed cycle:

$$
C=\{v_1,v_2,\ldots,v_k\}
$$

#### Step 4: Contract the cycle

Replace all nodes in the cycle by a single supernode.

For an edge entering the cycle:

$$
u\rightarrow v,\qquad v\in C
$$

use:

$$
w'(u\rightarrow C)
=
w(u\rightarrow v)
+
W_C
-
w(parent(v)\rightarrow v)
$$

where $W_C$ is the total score of the selected cycle edges.

For an edge leaving the cycle:

$$
v\rightarrow u,\qquad v\in C
$$

keep the maximum-scoring corresponding outgoing edge.

#### Step 5: Recursively solve the contracted graph

Run the same procedure on the contracted graph.

#### Step 6: Expand the cycle

Use the selected incoming edge to determine which cycle edge must be removed.

Restore the remaining cycle edges and return the final maximum spanning arborescence.

### Required functions

Complete:

```python
chu_liu_edmonds()

In [33]:

def select_best_incoming(nodes, edges, root="ROOT"):
    """
    Select the highest-scoring incoming edge for every non-root node.

    edges:
        dictionary {(head, dependent): score}
    """
    arcs = []

    for dep in nodes:
        if dep == root:
            continue

        candidates = [
            (head, dep)
            for head in nodes
            if head != dep and (head, dep) in edges
        ]
        if not candidates:
            continue

        arcs.append(max(candidates, key=lambda arc: edges[arc]))

    return arcs


def detect_cycle(arcs, root="ROOT"):
    """
    Return one directed cycle from arcs.
    Return None if no cycle exists.
    """
    head_of = {dep: head for head, dep in arcs}

    for start in head_of:
        path = []
        current = start
        while current in head_of and current != root:
            if current in path:
                return path[path.index(current):]
            path.append(current)
            current = head_of[current]

    return None


def contract_cycle(nodes, edges, selected_arcs, cycle, root="ROOT"):
    """
    Contract the detected cycle into a single supernode.

    Return:
        contracted_nodes
        contracted_edges
        mapping_information

    mapping_information is needed later to expand
    the contracted solution back to the original graph.
    """
    cycle_set = set(cycle)
    super_node = "C[" + "+".join(cycle) + "]"

    # the selected edge entering each node of the cycle, and their total weight
    inside = {dep: (head, dep) for head, dep in selected_arcs if dep in cycle_set}
    cycle_total = sum(edges[arc] for arc in inside.values())

    contracted_nodes = [n for n in nodes if n not in cycle_set] + [super_node]
    contracted_edges = {}

    entering = {}   # outside head -> the cycle node the collapsed edge really enters
    leaving = {}    # outside dependent -> the cycle node the collapsed edge really leaves

    for (head, dep), value in edges.items():
        if head in cycle_set and dep in cycle_set:
            continue                                # edges inside the cycle disappear

        if dep in cycle_set:                        # edge entering the cycle
            adjusted = value + cycle_total - edges[inside[dep]]
            key = (head, super_node)
            if key not in contracted_edges or adjusted > contracted_edges[key]:
                contracted_edges[key] = adjusted
                entering[head] = dep

        elif head in cycle_set:                     # edge leaving the cycle
            key = (super_node, dep)
            if key not in contracted_edges or value > contracted_edges[key]:
                contracted_edges[key] = value
                leaving[dep] = head

        else:                                       # edge untouched by the cycle
            contracted_edges[(head, dep)] = value

    mapping_information = {
        "super_node": super_node,
        "entering": entering,
        "leaving": leaving,
        "cycle_total": cycle_total,
    }

    return contracted_nodes, contracted_edges, mapping_information


def expand_cycle(
    selected_contracted_arcs,
    cycle,
    mapping_information,
    original_selected_arcs
):
    """
    Expand the contracted solution back to the original graph.

    The incoming edge selected for the supernode determines
    which cycle edge must be removed.
    """
    super_node = mapping_information["super_node"]
    entering = mapping_information["entering"]
    leaving = mapping_information["leaving"]

    cycle_set = set(cycle)
    inside = {dep: head for head, dep in original_selected_arcs if dep in cycle_set}

    final_arcs = []
    broken = None

    for head, dep in selected_contracted_arcs:
        if dep == super_node:
            broken = entering[head]                 # this cycle node gets a new head
            final_arcs.append((head, broken))
        elif head == super_node:
            final_arcs.append((leaving[dep], dep))
        else:
            final_arcs.append((head, dep))

    # keep every cycle edge except the one entering the node we came in through
    for node in cycle:
        if node != broken:
            final_arcs.append((inside[node], node))

    return final_arcs


def chu_liu_edmonds(tokens, score, root="ROOT"):
    """
    Implement the Chu-Liu/Edmonds maximum spanning
    arborescence algorithm from scratch.

    Return:
        total_score, final_arcs
    """

    # drop the edges a rooted tree can never use
    edges = {
        (head, dep): value
        for (head, dep), value in score.items()
        if dep != root and head != dep
    }

    # 1. highest-scoring incoming edge for every non-root node
    selected_arcs = select_best_incoming(tokens, edges, root)

    # 2. and 3. no cycle means the selected edges already form the tree
    cycle = detect_cycle(selected_arcs, root)
    if cycle is None:
        return sum(edges[arc] for arc in selected_arcs), selected_arcs

    # 4. contract the cycle into a supernode
    contracted_nodes, contracted_edges, mapping_information = contract_cycle(
        tokens, edges, selected_arcs, cycle, root
    )

    # 5. solve the smaller graph with the same procedure
    _, contracted_arcs = chu_liu_edmonds(contracted_nodes, contracted_edges, root)

    # 6. expand the supernode back into the original nodes
    final_arcs = expand_cycle(
        contracted_arcs, cycle, mapping_information, selected_arcs
    )

    return sum(edges[arc] for arc in final_arcs), final_arcs


# Test on the supplied graph
cle_score, cle_arcs = chu_liu_edmonds(tokens, score)

print("Chu-Liu/Edmonds score:", cle_score)
print("Chu-Liu/Edmonds arcs:", cle_arcs)

print(
    "Matches exhaustive MST score?",
    cle_score == best_score
)

Chu-Liu/Edmonds score: 26
Chu-Liu/Edmonds arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]
Matches exhaustive MST score? True


## Task 6: Inference-Based Learning with Structured Perceptron (5 marks)

In this task, you will train the MST dependency parser using **inference-based structured perceptron learning**.

The parser uses the current weight vector to score every possible dependency tree and selects the highest-scoring tree:

$$
\boxed{
G_{pred}
=
\arg\max_{G\in T(x)}
\mathbf{w}\cdot\Phi(x,G)
}
$$

where:

- $x$ = input sentence
- $T(x)$ = set of valid dependency trees for the sentence
- $\mathbf{w}$ = current feature-weight vector
- $\Phi(x,G)$ = feature vector of dependency tree $G$

The predicted tree is then compared with the gold tree.

If:

$$
G_{pred}\neq G_{gold}
$$

update the weights using:

$$
\boxed{
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
}
$$

Equivalently, for each feature:

$$
\boxed{
w_{new}[f]
=
w_{old}[f]
+
\Phi(x,G_{gold})[f]
-
\Phi(x,G_{pred})[f]
}
$$

### Training procedure

Your training loop must perform the following steps:

1. Initialize the feature weights.
2. Take one training sentence and its gold dependency tree.
3. Compute the score of candidate dependency arcs using the current weights.
4. Run your **own Chu-Liu/Edmonds implementation from Task 5** to obtain the predicted tree.
5. Compute the feature vector of the gold tree.
6. Compute the feature vector of the predicted tree.
7. Update the weights using the structured perceptron rule.
8. Repeat for all training examples.
9. Repeat the complete process for multiple epochs.

### Important

The training procedure must call your implementation from **Task 5**.

Do not use:

```python
networkx.maximum_spanning_arborescence()

In [34]:

from collections import Counter

# ---------------------------------------------------------
# Training data
# ---------------------------------------------------------

training_data = [

    {
        "tokens": ["ROOT", "John", "saw", "Mary"],
        "pos": {
            "ROOT": "ROOT",
            "John": "NOUN",
            "saw": "VERB",
            "Mary": "NOUN"
        },
        "gold": [
            ("ROOT", "saw"),
            ("saw", "John"),
            ("saw", "Mary")
        ]
    },

    {
        "tokens": ["ROOT", "Dogs", "chase", "cats"],
        "pos": {
            "ROOT": "ROOT",
            "Dogs": "NOUN",
            "chase": "VERB",
            "cats": "NOUN"
        },
        "gold": [
            ("ROOT", "chase"),
            ("chase", "Dogs"),
            ("chase", "cats")
        ]
    },

    {
        "tokens": ["ROOT", "Alice", "likes", "Bob"],
        "pos": {
            "ROOT": "ROOT",
            "Alice": "NOUN",
            "likes": "VERB",
            "Bob": "NOUN"
        },
        "gold": [
            ("ROOT", "likes"),
            ("likes", "Alice"),
            ("likes", "Bob")
        ]
    }
]


# ---------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------

def arc_features(tokens, pos, head, dep):
    """
    Return a list of feature names for one dependency arc.

    Required feature templates:
      1. head word
      2. dependent word
      3. head POS
      4. dependent POS
      5. direction
      6. distance
      7. head-dependent word pair
    """

    head_index = tokens.index(head)
    dep_index = tokens.index(dep)

    direction = "RIGHT" if head_index < dep_index else "LEFT"
    distance = abs(head_index - dep_index)

    return [
        f"HEAD_WORD={head}",
        f"DEP_WORD={dep}",
        f"HEAD_POS={pos[head]}",
        f"DEP_POS={pos[dep]}",
        f"DIRECTION={direction}",
        f"DISTANCE={distance}",
        f"PAIR={head}->{dep}",
    ]


def tree_feature_vector(tokens, pos, tree):
    """
    Sum the feature vectors of all arcs in a dependency tree.

    Return a Counter or dictionary:
        feature -> count
    """

    features = Counter()

    for head, dep in tree:
        features.update(arc_features(tokens, pos, head, dep))

    return features


# ---------------------------------------------------------
# Arc and tree scoring
# ---------------------------------------------------------

def score_arc(tokens, pos, head, dep, weights):
    """
    Score one dependency arc:

        score(h -> d) = w . f(h -> d)

    """

    return sum(weights[f] for f in arc_features(tokens, pos, head, dep))


def build_score_graph(tokens, pos, weights):
    """
    Construct the complete directed graph of candidate
    dependency arcs.

    Return:
        {(head, dependent): score}
    """

    graph = {}

    for head in tokens:
        for dep in tokens:
            if head == dep or dep == "ROOT":         # nothing points at ROOT
                continue
            graph[(head, dep)] = score_arc(tokens, pos, head, dep, weights)

    return graph


def score_tree(tokens, pos, tree, weights):
    """
    Return the total score of a dependency tree.
    """

    return sum(
        score_arc(tokens, pos, head, dep, weights)
        for head, dep in tree
    )


# ---------------------------------------------------------
# Structured perceptron update
# ---------------------------------------------------------

def perceptron_update(
    weights,
    gold_features,
    predicted_features,
    learning_rate=1.0
):
    """
    Structured perceptron update:

        w_new =
        w_old
        + eta * Phi(gold)
        - eta * Phi(predicted)
    """

    updated = Counter(weights)

    for feature, count in gold_features.items():
        updated[feature] += learning_rate * count

    for feature, count in predicted_features.items():
        updated[feature] -= learning_rate * count

    return updated


# ---------------------------------------------------------
# Inference-based training
# ---------------------------------------------------------

def train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
):
    """
    Train the MST parser using inference-based
    structured perceptron learning.

    For every epoch:

        1. Infer predicted tree using Task 5.
        2. Compare with gold tree.
        3. Update weights if prediction is incorrect.
        4. Record number of mistakes.
    """

    # Initialize all weights to zero.
    weights = Counter()

    history = []

    for epoch in range(num_epochs):

        mistakes = 0

        for example in training_data:

            tokens = example["tokens"]
            pos = example["pos"]
            gold_tree = example["gold"]

            # ---------------------------------------------
            # Build arc scores using current weights.
            # ---------------------------------------------

            score_graph = build_score_graph(
                tokens,
                pos,
                weights
            )

            # ---------------------------------------------
            # Run the Task 5 implementation.
            # ---------------------------------------------

            predicted_score, predicted_tree = chu_liu_edmonds(
                tokens,
                score_graph
            )

            # ---------------------------------------------
            # Compare predicted and gold trees.
            # ---------------------------------------------

            if set(predicted_tree) != set(gold_tree):

                mistakes += 1

                # -----------------------------------------
                # Compute gold feature vector.
                # -----------------------------------------

                gold_features = tree_feature_vector(
                    tokens,
                    pos,
                    gold_tree
                )

                # -----------------------------------------
                # Compute predicted feature vector.
                # -----------------------------------------

                predicted_features = tree_feature_vector(
                    tokens,
                    pos,
                    predicted_tree
                )

                # -----------------------------------------
                # Update weights.
                # -----------------------------------------

                weights = perceptron_update(
                    weights,
                    gold_features,
                    predicted_features,
                    learning_rate
                )

        history.append(mistakes)

        print(
            f"Epoch {epoch + 1}: "
            f"{mistakes} mistake(s)"
        )

    return weights, history

In [35]:
# Run the complete inference-based training

final_weights, training_history = train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
)

print("\nTraining history:")
for epoch, mistakes in enumerate(training_history, start=1):
    print(
        f"Epoch {epoch}: "
        f"{mistakes} mistake(s)"
    )

print("\nLearned weights:")
for feature, weight in sorted(final_weights.items()):
    print(f"{feature}: {weight}")

Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Training history:
Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Learned weights:
DEP_POS=NOUN: 0.0
DEP_POS=VERB: 0.0
DEP_WORD=John: 0.0
DEP_WORD=Mary: 0.0
DEP_WORD=saw: 0.0
DIRECTION=LEFT: 1.0
DIRECTION=RIGHT: -1.0
DISTANCE=1: 1.0
DISTANCE=2: 0.0
DISTANCE=3: -1.0
HEAD_POS=ROOT: -2.0
HEAD_POS=VERB: 2.0
HEAD_WORD=ROOT: -2.0
HEAD_WORD=saw: 2.0
PAIR=ROOT->John: -1.0
PAIR=ROOT->Mary: -1.0
PAIR=ROOT->saw: 0.0
PAIR=saw->John: 1.0
PAIR=saw->Mary: 1.0


## Task 6: Training Analysis

### Q1. Prediction Errors

For each epoch, report the number of incorrectly predicted trees.

| Epoch | Number of Mistakes |
|------:|-------------------:|
| 1 | 1 |
| 2 | 0 |
| 3 | 0 |
| 4 | 0 |
| 5 | 0 |

The only mistake happened on the first sentence of the first epoch. With every weight starting at zero, all arcs score 0 and the parser has no reason to prefer one tree over another, so it returned

```text
ROOT -> John
ROOT -> saw
ROOT -> Mary
```

instead of the gold tree

```text
ROOT -> saw
saw  -> John
saw  -> Mary
```

### Q2. Weight Updates

Features whose weights increased:

| Feature | Final weight |
|---|---:|
| `HEAD_WORD=saw` | +2 |
| `HEAD_POS=VERB` | +2 |
| `PAIR=saw->John` | +1 |
| `PAIR=saw->Mary` | +1 |
| `DIRECTION=LEFT` | +1 |
| `DISTANCE=1` | +1 |

These features are counted more often in the gold tree than in the predicted tree, so `Phi(gold) - Phi(pred)` is positive for them. In the gold tree `saw` is the head of two arcs and in the prediction it is the head of none, which is why `HEAD_WORD=saw` and `HEAD_POS=VERB` both move by +2. The arcs `saw -> John` and `saw -> Mary` exist only in the gold tree, so their pair features gain +1 each. `DIRECTION=LEFT` and `DISTANCE=1` rise for the same reason, because the gold tree contains the short leftward arc `saw -> John` and the prediction contains no leftward arc at all.

### Q3. Negative Updates

Features whose weights decreased:

| Feature | Final weight |
|---|---:|
| `HEAD_WORD=ROOT` | -2 |
| `HEAD_POS=ROOT` | -2 |
| `PAIR=ROOT->John` | -1 |
| `PAIR=ROOT->Mary` | -1 |
| `DIRECTION=RIGHT` | -1 |
| `DISTANCE=3` | -1 |

These features are counted more often in the predicted tree than in the gold tree, so the difference is negative. The prediction hangs all three words directly under `ROOT`, giving `ROOT` three head counts against one in the gold tree, which produces the -2 on `HEAD_WORD=ROOT` and `HEAD_POS=ROOT`. The arcs `ROOT -> John` and `ROOT -> Mary` appear only in the prediction, so their pair features lose 1 each. `DISTANCE=3` drops because only the wrong tree contains an arc that long.

It is worth noting that features such as `DEP_WORD=John` and `DEP_POS=NOUN` stayed at 0. Both trees give every word exactly one head, so those counts are identical in the two feature vectors and cancel out.

### Q4. Effect of the Perceptron Update

The score of a tree is the dot product `w . Phi(x, G)`, so a tree gets a higher score when the features it contains have larger weights. The update

$$
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
$$

adds weight to every feature that the gold tree uses more and takes weight away from every feature that the wrong tree uses more. Under the new weights the gold tree therefore scores higher than before and the predicted tree scores lower than before. The gap between them closes by exactly the squared distance between the two feature vectors, so the next time Chu-Liu/Edmonds runs on this sentence the gold tree is a stronger candidate and the mistaken tree is a weaker one. Features that both trees share are left untouched, which keeps the update focused only on the part of the structure that was actually wrong.

### Q5. Training Behaviour

Yes. The count dropped from 1 mistake in epoch 1 to 0 mistakes in every later epoch.

The single update was enough for two reasons. First, it directly fixed the sentence it came from, because `ROOT` became an expensive head and `saw` became a cheap one, so re-parsing `ROOT John saw Mary` now returns the gold tree. Second, the update generalised to the other two sentences before they were even seen. All three sentences have the same NOUN VERB NOUN shape and the same gold structure, and the features `HEAD_POS=VERB`, `HEAD_POS=ROOT`, `DIRECTION=LEFT` and `DISTANCE=1` do not mention any particular word. So the weights learned from `John saw Mary` already scored the correct tree highest for `Dogs chase cats` and `Alice likes Bob`, and the parser never made a mistake on them. Once an epoch passes with no mistakes the weights stop changing, which is why the remaining epochs stay at 0.